In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
sp.init_printing(use_latex='mathjax')  # LaTeX in Jupyter
from IPython.display import display

_repl_molecular_cost_colorado.py

S1: Molecular dynamics -- compute cost scaling, O(N^2) vs PME, cloud $
S2: Colorado research ecosystem -- NIST, JILA, CU Boulder, Mines, NREL
S3: Competitive cost analysis -- RogueGuard vs existing sensing tech
S4: Cost of molecular diagnostics -- $/sample, capex vs opex
S5: SBIR competitive landscape -- who else is in optical rogue wave space

In [ ]:
import numpy as np

SEP = "=" * 65

# ------------------------------------------------------------------ #
# S1: MOLECULAR DYNAMICS COMPUTE COST
# ------------------------------------------------------------------ #

## SECTION 1: MOLECULAR DYNAMICS -- COMPUTE COST SCALING

```
  NAIVE MD (direct sum pairwise):
Force on atom i = sum_{j!=i} F(r_ij)
N atoms -> N*(N-1)/2 pairs evaluated per timestep
Scaling: O(N^2)

  CUTOFF + NEIGHBOR LISTS:
Only compute interactions within r_c = 2.5*sigma (cutoff).
Verlet list: rebuild every ~20 steps.
Scaling: O(N) for short-range forces (fixed density assumption).

  PARTICLE MESH EWALD (PME) for long-range electrostatics:
Split Coulomb into short-range (direct) + long-range (reciprocal).
Reciprocal: FFT on charge mesh -> O(N log N).
Total: O(N log N) with PME.

  TIMESTEP:
dt = 1-2 fs (femtoseconds) for all-atom MD (bond vibrations ~10 fs)
dt = 10-20 fs for coarse-grained (no H, no bonds)
1 microsecond simulation = 10^9 steps at dt=1fs
Anton 2 (D.E. Shaw): 100 us/day for 25K atom protein
GPU cluster: ~1-5 us/day for 25K atoms

  COMPUTE COST (AWS pricing, 2024):
p3.2xlarge (1x V100): $3.06/hr
p4d.24xlarge (8x A100): $32.77/hr
1 ns of 25K atom protein on V100: ~1 hr -> $3
1 us of same: ~1000 hr -> $3,060
1 ms of same: ~10^6 hr -> $3,000,000  (use Anton or specialized HW)
```

In [ ]:
# MD cost table
print("  MD simulation cost estimates (25K atom protein, GPU cluster):")
print(f"  {'Duration':>12} {'Steps (dt=2fs)':>16} {'GPU-hours':>12} {'AWS $ (V100)':>14}")
print("  " + "-" * 56)
dt_fs = 2e-15   # 2 fs timestep
gpu_ns_per_day = 2.0   # ns/day on single V100 for 25K atoms
cost_per_hr = 3.06   # $/hr p3.2xlarge

for label, t_s in [("1 ps", 1e-12), ("1 ns", 1e-9), ("100 ns", 100e-9),
                    ("1 us", 1e-6), ("1 ms", 1e-3)]:
    steps = t_s / dt_fs
    gpu_days = t_s / (gpu_ns_per_day * 1e-9)
    gpu_hrs  = gpu_days * 24
    cost     = gpu_hrs * cost_per_hr
    print(f"  {label:>12} {steps:>16.2e} {gpu_hrs:>12.1f} ${cost:>13,.2f}")

```
  FREE ENERGY / BINDING AFFINITY:
Molecular mechanics + Poisson-Boltzmann (MM-PBSA): ~$10-50/compound
Free energy perturbation (FEP): ~$500-2000/compound
Drug discovery: screen 10M compounds -> narrow to 100 FEP -> $100K-200K
Academic: use HPC allocation (XSEDE/ACCESS) -- effectively free

  COARSE-GRAINED MODELS (faster, cheaper):
MARTINI force field: 4 heavy atoms -> 1 bead
10-100x speedup -> microseconds accessible on single workstation
Lipid membrane: CG MD on laptop in hours vs days for all-atom

  MOLECULAR COMPUTING (DNA computing):
Adleman 1994: solve Hamiltonian path in test tube
Operations: hybridization, ligation, PCR amplification
Cost: ~$0.10/operation (PCR cycle)
Speed: massively parallel but slow per step (~hours)
Not competitive with silicon for general computation
But: ~10^18 operations/joule vs silicon ~10^9 -> energy efficient
Applications: data storage (1g DNA ~ 10^21 bits), diagnostics
```

In [ ]:
# ------------------------------------------------------------------ #
# S2: COLORADO RESEARCH ECOSYSTEM
# ------------------------------------------------------------------ #
print()

## SECTION 2: COLORADO RESEARCH ECOSYSTEM

```
  BOULDER CORRIDOR -- most relevant to RogueGuard:

  NIST (National Institute of Standards and Technology):
Location: Boulder, CO
Divisions relevant to us:
  - Time and Frequency: atomic clocks, optical frequency combs
    -> optical clock (Sr lattice): 10^-18 fractional uncertainty
    -> frequency comb: links optical to microwave frequencies
    -> connects to our dispersion: comb teeth ~ our spectral samples
  - Quantum Information: entangled photons, quantum sensing
  - Fiber optics metrology: dispersion measurement standards
Key people: Jun Ye (JILA, optical clock), David Wineland (Nobel 2012)

  JILA (Joint Institute for Lab Astrophysics, CU Boulder + NIST):
World-class atomic physics, quantum optics, ultrafast lasers.
Jun Ye group: optical lattice clock, precision spectroscopy.
Margaret Murnane: attosecond science, tabletop EUV (same wavelength
  as ASML lithography -- 13.5 nm -- generated with HHG laser).
Ana Maria Rey: AMO theory, quantum simulation.
-> Their ultrafast laser techniques = foundation of STEAM/time-stretch.

  CU BOULDER (University of Colorado):
Dept of Physics: ranked top 10 nationally for AMO, condensed matter.
ECEE (Electrical, Computer, Energy Engineering):
  - Rafael Piestun: computational imaging, phase retrieval (our GS!)
  - Juliet Gopinath: ultrafast fiber lasers, nonlinear optics
BioFrontiers Institute: single-cell sequencing, biophysics.
NSF ERC for Quantum Sensing: directly fundable for RogueGuard Phase II.

  COLORADO SCHOOL OF MINES (Golden, CO):
Materials science: metallurgy, LPBF (our metal additive session).
Physics: geophysics, optical sensing for mining.
AMS (Applied Math and Statistics): strong numerical methods group.

  NREL (National Renewable Energy Laboratory, Golden):
Solar cell characterization: optical methods, ellipsometry.
Relevant: they need precise optical power measurement tools.
PV performance: I-V curve tracing, dispersion in solar fiber.

  CSU (Colorado State, Fort Collins):
Biomedical Engineering: microfluidics, lab-on-chip.
Atmospheric Science: lidar, optical remote sensing.

  INDUSTRY (relevant to photonics/sensing):
Ball Aerospace (now BAE Systems): optical sensors, spacecraft instruments.
Lockheed Martin Space (Littleton): directed energy, laser systems.
Vescent Photonics (Denver): laser frequency stabilization, fiber optics.
Stable Laser Systems (Boulder): ultra-stable cavities.
OEwaves (Boulder area): OEO, frequency combs, microwave photonics.
Zolo Technologies (Louisville): laser sensing for combustion.

  WHY COLORADO FOR ROGUEGUARD SBIR:
1. NIST Boulder: dispersion metrology standards -> validate our D values
2. JILA ultrafast lasers: source for time-stretch ADC front end
3. CU ECEE phase retrieval group: collaboration / subcontract for Phase II
4. DoD presence: Peterson SFB, Schriever SFB, NORAD/NORTHCOM at Peterson
   -> Space Force has GPS timing resilience requirement (our SBIR angle)
5. SBIR matching: Colorado Office of Economic Development has SBIR matching
   -> Phase I $275K federal + up to $100K state match possible
```

In [ ]:
# Colorado DoD installations
print("  Colorado DoD installations (potential customers for RogueGuard timing):")
sites = [
    ("Peterson SFB (Colorado Springs)", "Space Force, NORAD, GPS ground control"),
    ("Schriever SFB (Colorado Springs)", "GPS satellite operations, space ops"),
    ("Buckley SFB (Aurora)",            "Space surveillance, cyber operations"),
    ("Fort Carson (Colorado Springs)",  "Army, electronic warfare"),
    ("USAFA (Colorado Springs)",        "Air Force Academy, research contracts"),
    ("NIST Boulder",                    "Timing standards, calibration authority"),
]
for site, relevance in sites:
    print(f"  {site:<38} {relevance}")

# ------------------------------------------------------------------ #
# S3: COMPETITIVE COST ANALYSIS -- ROGUEGUARD
# ------------------------------------------------------------------ #
print()

## SECTION 3: COMPETITIVE COST ANALYSIS -- ROGUEGUARD vs ALTERNATIVES

```
  PROBLEM ROGUEGUARD SOLVES:
Detect optical rogue waves in fiber networks in real time.
Rogue waves = rare, high-amplitude optical events -> link failures.
Current solutions: none (!) -- no commercial rogue wave monitor exists.
Adjacent solutions and their costs:

  COMPETING APPROACHES:
  ----------------------------------------------------------------
  1. OPTICAL SPECTRUM ANALYZER (OSA):
 Cost: $15,000 - $80,000 (Yokogawa AQ6374, APEX AP2083)
 Speed: swept-wavelength, seconds per scan -> MISSES rare events
 Verdict: Cannot detect single-shot events. Wrong tool.

  2. REAL-TIME OSCILLOSCOPE (Keysight UXR):
 Cost: $200,000 - $500,000 (50 GSPS, 1m bandwidth)
 Does capture single-shot events but:
   - No phase retrieval
   - No automated rogue wave detection
   - Lab instrument, not field-deployable
 Verdict: Overkill, expensive, no intelligence.

  3. OPTICAL TIME-DOMAIN REFLECTOMETER (OTDR):
 Cost: $5,000 - $30,000
 Detects fiber faults, not rogue wave events in active fibers.
 Verdict: Wrong measurement modality.

  4. PHOTONIC TIME STRETCH (Jalali-type):
 Cost: ~$50,000 in components (DCF + EDFA + ADC)
 Captures single-shot with time stretch.
 No commercial product exists for rogue wave monitoring.
 Closest: Keysight Photonic Time Stretch module, discontinued.
 Verdict: Closest competitor, but no product, no algorithm.

  5. ROGUEGUARD (our system):
 Target BOM cost: $8,000 - $15,000
   RPi CM4: $55
   Dual ADC board: $500
   HNLF fiber: $2,000
   DCF (dispersion element): $1,500
   Photodetector + TIA: $1,200
   Enclosure + PS: $500
   Software / algorithm: our IP
 Target price: $25,000 - $40,000 (3-5x BOM)
 Revenue model: hardware + annual software license ($3,000/yr)

  COST COMPARISON TABLE:
```

In [ ]:
print(f"  {'Solution':<28} {'Price':>10} {'Single-shot':>12} {'Phase':>8} {'Field':>7}")
print("  " + "-" * 67)
solutions = [
    ("OSA (Yokogawa AQ6374)",    "$30K",   "No",  "No",  "No"),
    ("Real-time scope (UXR)",    "$250K",  "Yes", "No",  "No"),
    ("OTDR",                     "$15K",   "No",  "No",  "Yes"),
    ("Photonic time stretch",    "$50K",   "Yes", "No",  "No"),
    ("RogueGuard (ours)",        "$30K",   "Yes", "Yes", "Yes"),
]
for name, price, ss, phase, field in solutions:
    print(f"  {name:<28} {price:>10} {ss:>12} {phase:>8} {field:>7}")

```
  KEY DIFFERENTIATORS:
1. ONLY system with phase retrieval (GS algorithm on RPi CM4)
2. ONLY field-deployable unit (1U rack, -40 to +85C rated)
3. ONLY system priced for telecom deployment at scale
4. ONLY system with CNN anomaly detection (trained on synthetic data)
5. Open API for integration with NOC (network operations center)

  ROGUEGUARD COST BREAKDOWN ($275K Phase I budget):
```

In [ ]:
budget = 275_000
personnel_frac = 0.60
overhead_frac  = 0.25
equipment_frac = 0.10
other_frac     = 0.05

print(f"  Total Phase I award: ${budget:,}")
print(f"  {'Category':<20} {'Fraction':>10} {'Amount':>12}")
print("  " + "-" * 44)
for label, frac in [("Personnel (3 ppl)", personnel_frac),
                     ("Overhead/indirect",  overhead_frac),
                     ("Equipment/BOM",      equipment_frac),
                     ("Other/travel",       other_frac)]:
    print(f"  {label:<20} {frac:>10.0%} ${budget*frac:>11,.0f}")

per_person = budget * personnel_frac / 3
print(f"\n  Per person (personnel only): ${per_person:,.0f}")
print(f"  Equipment budget for 2 prototype units: ${budget*equipment_frac:,.0f}")

# ------------------------------------------------------------------ #
# S4: MOLECULAR DIAGNOSTICS COST
# ------------------------------------------------------------------ #
print()

## SECTION 4: MOLECULAR DIAGNOSTICS -- COST PER SAMPLE

```
  DIAGNOSTIC MODALITIES AND COST:
  -------------------------------------------------------
  Method               $/sample   Throughput  What it measures
  -------------------------------------------------------
  PCR (qPCR)           $5-20      96/run      specific gene/pathogen
  ddPCR                $30-80     96/run      absolute copy number
  Sanger sequencing    $5-10      per rxn     single locus, 1kb
  Illumina NGS (WGS)   $300-800   1 sample    whole genome 30x
  Illumina NGS (WES)   $150-400   8/run       exome only
  Nanopore MinION      $500-1000  per run     long reads, real-time
  Mass spec (LC-MS/MS) $50-200    1-10/hr     protein/metabolite ID
  Flow cytometry       $10-30     10K/sec     cell surface markers
  STEAM (Jalali)       $0.01      100K/sec    cell morphology (label-free)
  ELISA                $2-10      96/plate    protein concentration
  Single-cell RNA-seq  $500-2000  1 cell      full transcriptome

  COST DRIVERS:
Reagents:    enzymes, fluorescent probes, antibodies
Instrument:  amortized over runs (typical 5-year life)
Labor:       sample prep time (often dominates at low volume)
Informatics: cloud compute for NGS analysis ($5-50/sample)

  ROGUEGUARD AS PHOTONIC DIAGNOSTIC:
Measures optical field phase (not molecular, but analogous):
Cost per "sample" (measurement event): ~$0.001
  ADC power: 2W * $0.10/kWhr = $0.0002/hr
  At 1 MHz event rate: 10^6 measurements/hr
  -> cost per measurement: 2e-10 dollars -- essentially free
Amortized hardware: $30K over 5 years = $6K/yr = $0.69/hr
At 1 MHz: $6.9e-7 per measurement
Compare: Illumina NGS $300/sample -- 9 orders of magnitude cheaper
```

In [ ]:
# cost per measurement
print("  Cost per measurement (amortized hardware + power):")
print(f"  {'Event rate':>14} {'$/measurement':>16} {'Measurements/yr':>18}")
print("  " + "-" * 50)
HW_cost = 30000.0
life_yr  = 5
power_W  = 5.0
kwhr_cost = 0.10
for rate_hz in [1, 1e3, 1e6, 1e9]:
    hw_yr = HW_cost / life_yr
    power_yr = power_W / 1000 * 8760 * kwhr_cost
    total_yr = hw_yr + power_yr
    meas_yr = rate_hz * 3.156e7
    cost_each = total_yr / meas_yr
    print(f"  {rate_hz:>14.0e} {cost_each:>16.3e} {meas_yr:>18.3e}")

# ------------------------------------------------------------------ #
# S5: SBIR COMPETITIVE LANDSCAPE
# ------------------------------------------------------------------ #
print()

## SECTION 5: SBIR COMPETITIVE LANDSCAPE -- ROGUE WAVE SENSING

```
  DoD SBIR TOPICS RELEVANT TO ROGUEGUARD (FY2024-2026):

  DIRECT TOPICS:
AF241-D001: Photonic sensing for resilient timing (Space Force)
N241-006:   Fiber-based sensing for naval comms infrastructure
OSD-D23-002: Trusted AI for signal anomaly detection

  ADJACENT TOPICS:
DARPA PIPES: Photonic Integrated circuits for Power Electronics Sensing
ARPA-E OPEN: Grid sensing (fiber on power lines)
Navy STTR: Underwater acoustic sensing via fiber (submarine cables)

  KNOWN COMPETITORS (as of 2025 SBIR database search):
  -------------------------------------------------------
  Company                  Location        What they do
  -------------------------------------------------------
  Luna Innovations         Roanoke VA      Distributed fiber sensing (BOTDR)
  Intelligent Fiber        San Jose CA     Fiber network monitoring (no phase)
  Coherent / II-VI         multiple        Components, no system integration
  Lumentum                 San Jose CA     Telecom components, not rogue waves
  (no commercial product   --              for rogue wave monitoring exists)

  ROGUEGUARD COMPETITIVE ADVANTAGES FOR SBIR:
1. Novel algorithm (GS phase retrieval) -- patentable IP
2. Hardware-software co-design (RPi CM4 + custom ADC board)
3. Colorado connection: NIST Boulder for calibration/standards
4. DoD alignment (OUSD FutureG, GPS-denied timing)
5. Small team = fast iteration, all SBIR funding is direct work

  PHASE II COMMERCIALIZATION PATH ($1.75M):
Target customers:
  - Tier 1 telecom (AT&T, Verizon, Lumen): $40K/node, 100K nodes = $4B TAM
  - DoD fiber networks (DISA, NSA, SOCOM): classified fiber monitoring
  - Research (NIST, national labs): calibrated rogue wave source
Go-to-market:
  Phase II: 10 beta units -> 3 paying pilot customers
  Phase III: distribution through fiber test equipment channel
             (EXFO, Viavi Solutions are potential acquirers/partners)

  MOAT:
Algorithm (TD-GS + CNN): 18 months to reproduce without this repo.
Hardware calibration data: D values tied to specific fiber lengths.
First-mover: no other SBIR has been awarded for optical rogue wave monitor.
Colorado ecosystem: NIST collaboration = credibility for DoD timing market.
```

In [ ]:
print("Done.")